In [70]:
print("hello1")

hello1


## 저장 버전 1

In [111]:
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 3. 수집된 데이터를 저장할 파일 이름 입력 받기
f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본값:C:\\py_temp\\추출 후 파일 저장\\):")
if f_dir == '' :
    f_dir="C:\\py_temp\\추출 후 파일 저장\\"

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url ='https://eungdapso.seoul.go.kr/main.do'
driver.get(url)
driver.maximize_window()
time.sleep(2)

#Step 5. 자동으로 검색어 입력 후 조회하기
driver.find_element(By.XPATH, '//div[text()="시장에게 바란다!"]').click() #div 중에서 text가 "시장에게 바란다!"라는 걸 찾고 클릭
time.sleep(3)

## 저장 버전 2

In [112]:
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import os
from datetime import datetime

# 1. 저장할 기본 폴더 경로 설정 (경로 마지막에 꼭 \\ 를 붙여주세요)
save_path = "C:\\py_temp\\추출 후 파일 저장\\"

# 해당 폴더가 컴퓨터에 없으면 자동으로 생성해 주는 안전장치
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2. '검색어_현재시간' 형태로 파일명 자동 생성
now = datetime.now().strftime("%Y%m%d_%H%M") # 파일명 중복을 막기 위한 현재 시간(년월일_시분)

query_txt = input("2.파일을 저장할 폴더명만 쓰세요")
ft_name = f"{save_path}{query_txt}_{now}.txt"
fc_name = f"{save_path}{query_txt}_{now}.csv"
fx_name = f"{save_path}{query_txt}_{now}.xlsx"

print(f"👉 수집된 데이터는 [{save_path}] 폴더에 자동 저장됩니다.")

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url ='https://eungdapso.seoul.go.kr/main.do'
driver.get(url)
driver.maximize_window()
time.sleep(2)

#Step 5. 자동으로 검색어 입력 후 조회하기
driver.find_element(By.XPATH, '//div[text()="시장에게 바란다!"]').click() #div 중에서 text가 "시장에게 바란다!"라는 걸 찾고 클릭
time.sleep(3)

👉 수집된 데이터는 [C:\py_temp\추출 후 파일 저장\] 폴더에 자동 저장됩니다.


In [113]:
#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력 받기
import math
total_cnt = soup_1.find('div','rp_lkment').find('span','rplk_count').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
page_cnt = math.ceil(cnt / 10)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(cnt,page_cnt))
print("\n")

키워드 이제 그만 (으)로 총 340건 건의 학위논문이 검색되었습니다
58 건의 데이터를 수집하기 위해 6 페이지의 게시물을 조회합니다.




In [114]:
#Step 9. 데이터 수집하기
no2=[] # 게시글 번호 컬럼
title2=[ ] # 게시글 제목 컬럼
opend2=[] # 공개일
story2=[ ] # 상담내역

no = 1 # 게시글 번호 초기값

for a in range(1,page_cnt+1) :
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %a)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    time.sleep(3)
    content_list = soup.find('div','rp_tb_wrap').find('tbody').find_all('tr')

    for i in content_list:
        # 민원 제목 체크하기
        try:
            title=i.find('a','rp_inlink').get_text().strip()
        except :
            continue
        else :
            f = open(ft_name, 'a' , encoding="UTF-8")		#open : 준비 / a 쓰면 추가, w 쓰면 덮어쓰기(w쓰면 100건 크롤링해도 1건만 남음)
            # 1.게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.번호 : %s" %no)
            f.write('\n'+'1.번호:' + str(no))		# write : 쓰기 / 글자, 숫자면 안 맞아서 에러남. 그래서 str 사용하여 글자로 바꿈

            # 2. 민원 제목
            title2.append(title.strip())
            print("2.제목 : %s" %title.strip())
            f.write('\n' + '2.제목:' + title)
            
            # 3. 공개일
            try :
                opend=i.find("td","wid03").find('span').get_text().replace("\n","").strip()
            except :
                opend='공개일이 없습니다'
                opend2.append(opend)
                print("3.공개일 : %s" %opend)
            else :
                opend2.append(opend)
                print("3.공개일 : %s" %opend)
                f.write('\n' + '3.공개일:' + opend)

            time.sleep(1)

            # 상세정보로 들어가야 함
            url_1 = i.find('div','rp_inlink_item').find('a')['href'] #이게 포인트
            driver.execute_script(url_1) #URL아니고 자바 언어이기에 execute_script사용
            time.sleep(2)

            # 4. 상담내용
            html_1 = driver.page_source
            soup_1 = BeautifulSoup(html_1, 'html.parser')
            time.sleep(2)
            try :
                story=soup_1.find('div','restd_incont').find('span').get_text().strip()
            except :
                story = '상담내용이 없습니다'
                print("4.상담내용 : %s" %story.strip())
                story2.append(story.strip())
            else :
                story2.append(story.strip())
                print("4.상담내용 : %s" %story.strip())
                f.write('\n' + '4.상담내용:' + story + '\n')
                f.close( )			# close : 파일 저장

            driver.back() # 이전 페이지로 돌아가기

            time.sleep(2)

            no += 1

            if no > cnt :
                break

    a += 1
    b = str(a)

    try:
        driver.find_element(By.LINK_TEXT, b).click()
        time.sleep(1)

    except:
        driver.find_element(By.CLASS_NAME, "next_go").click()
        time.sleep(2)

print("요청하신 작업이 모두 완료되었습니다")



1 페이지 내용 수집 시작합니다 =======================


1 번째 정보를 추출하고 있습니다============
1.번호 : 1
2.제목 : 위례과천선 법조타운-북위례역(송파레이크파크호반써밋1,2차 사거리)-거여역 연장 강력 촉구
3.공개일 : 2026-02-25
4.상담내용 : 안녕하세요.

저는 서울시 위례 송파레이크파크호반써밋1,2차 아파트 주민입니다.

2024년 11월 8일 국토교통부에서 발표한 위례과천선 민자적격성 조사 통과 보도참고자료에 의하면 위례과천선 동쪽 구간은 송파구 법조타운으로 연결되는 노선으로 확정되었습니다. 위례 신도시의 교통불편을 해소하기 위해 계획되었던 위례과천선은 최초 노선안인 북위례지역을 경유하여 거여역으로 연결되는 노선이 아닌 송파구 법조타운으로 연결되는 노선으로 확정되었습니다. 
위례과천선은 위례 신도시의 교통불편을 해소하기 위해 계획되었고 위례 신도시 아파트 분양 납부금에는 교통 불모지였던 위례 신도시의 교통망을 확충하기 위한 교통분담금이 포함되어 있습니다. 따라서 위례를 거쳐가거나 위례와 이어지지 않는 위례과천선은 그 어떤 이유로도 납득할 수 없고 절대로 수용할 수 없습니다. 위례과천선 신설에 따른 수혜의 직접 대상이 되어야 하는 위례 신도시 주민들은 정작 교통망 확충의 혜택을 받지 못하게 되어 위례 신도시 주민으로서 너무나 참담하고 부당한 처사라고 생각합니다.
그러므로 위례과천선 동쪽 구간은 위례과천선 최초 원안을 반영하여 송파구 법조타운이 아니라 송파구 법조타운에서 북위례지역을 경유하여 거여역으로 연결되는 노선으로 확정되어야 합니다. 위례과천선 북위례지역 경유를 통한 거여역 연장은 한시도 지체할 수 없는 긴급하고 중대한 사안이며 다음과 같은 이유에서 즉시 확정되어야 합니다.

1. 북위례와 거여/마천 일대에 신축과 재건축 아파트가 속속 준공되고 입주가 완료되면서 거주 인구가 대폭 증가하고 그에 따라 교통수요는 폭발적으로 증가하였지만 교통망 확충은 한참 부족합니다. 북위례 지역의 A1-1(11단

In [115]:

# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
# 현재 날짜와 시간으로 폴더 만들고 파일 이름 설정하기
import os

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'응답소'+'-'+s+'-'+'민원')

ft_name = f_dir+'응답소'+'-'+s+'-'+'민원'+'\\'+'응답소'+'-'+s+'-'+'민원'+'.txt'
fc_name = f_dir+'응답소'+'-'+s+'-'+'민원'+'\\'+'응답소'+'-'+s+'-'+'민원'+'.csv'
fx_name = f_dir+'응답소'+'-'+s+'-'+'민원'+'\\'+'응답소'+'-'+s+'-'+'민원'+'.xlsx'

# 데이터 프레임 생성 후 xlsx, csv 형식으로 저장하기
import pandas as pd

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['공개일']=pd.Series(opend2)
df['상담내역']=pd.Series(story2)

# xls 형태로 저장하기
df.to_excel(fx_name,index=False)

# csv 형태로 저장하기
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

요청하신 데이터 수집 작업이 정상적으로 완료되었습니다


## 강사님 정답지

In [ ]:
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import random
import time

#Step 3. 수집된 데이터를 저장할 파일 이름 입력 받기
f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본값:C:\\py_temp\\추출 후 파일 저장\\):")
if f_dir == '' :
    f_dir="C:\\py_temp\\추출 후 파일 저장\\"

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s_time = time.time( ) #총 얼마걸리는지 알아냄
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get('https://eungdapso.seoul.go.kr/')
driver.maximize_window()
time.sleep(random.randrange(2,5))

#Step 5. 자동으로 검색어 입력 후 조회하기
driver.find_element(By.XPATH, '//*[@id="content"]/div[2]/a/div/div/div[1]/div[1]').click() #div 중에서 text가 "시장에게 바란다!"라는 걸 찾고 클릭
time.sleep(3)

In [ ]:
#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력 받기
import math
total_cnt = soup_1.find('div','rp_lkment').find('span','rplk_count').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
page_cnt = math.ceil(cnt / 10)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(cnt,page_cnt))
print("\n")

In [ ]:
#Step 9. 데이터 수집하기
no2=[] # 게시글 번호 컬럼
title2=[ ] # 게시글 제목 컬럼
opend2=[] # 공개일
story2=[ ] # 상담내역

no = 1 # 게시글 번호 초기값

for x in range(1,page_cnt+1) :
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %a)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    time.sleep(3)
    content_list = soup.find('div','rp_tb_wrap').find('tbody').find_all('tr')

    for i in content_list:
        # 민원 제목 체크하기
        try:
            title=i.find('a','rp_inlink').get_text().strip()
        except :
            continue
        else :
            f = open(ft_name, 'a' , encoding="UTF-8")		#open : 준비 / a 쓰면 추가, w 쓰면 덮어쓰기(w쓰면 100건 크롤링해도 1건만 남음)
            # 1.게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.번호 : %s" %no)
            f.write('\n'+'1.번호:' + str(no))		# write : 쓰기 / 글자, 숫자면 안 맞아서 에러남. 그래서 str 사용하여 글자로 바꿈

            # 2. 민원 제목
            title2.append(title.strip())
            print("2.제목 : %s" %title.strip())
            f.write('\n' + '2.제목:' + title)
            
            # 3. 공개일
            try :
                opend=i.find("td","wid03").find('span').get_text().replace("\n","").strip()
            except :
                opend='공개일이 없습니다'
                opend2.append(opend)
                print("3.공개일 : %s" %opend)
            else :
                opend2.append(opend)
                print("3.공개일 : %s" %opend)
                f.write('\n' + '3.공개일:' + opend)

            time.sleep(1)




            # 상세정보로 들어가야 함         
            driver.find_element(By.XPATH, '//*[@id="content"]/div[2]/div[4]/table/tbody/tr[%s]/td[2]/div/a' %i).click() #url이 없이 자바스크립트로 있기에 XPATH로 가야함
            time.sleep(2)

            # 4. 상담내용
            # 여기서 파싱이 중요(상세정보로 가져왔기에)
            html_1 = driver.page_source
            soup_1 = BeautifulSoup(html_1, 'html.parser')
            time.sleep(2)
            try :
                story=soup_1.find('div','restd_incont').find('span').get_text().strip()
            except :
                story = '상담내용이 없습니다'
                print("4.상담내용 : %s" %story.strip())
                story2.append(story.strip())
            else :
                story2.append(story.strip())
                print("4.상담내용 : %s" %story.strip())
                f.write('\n' + '4.상담내용:' + story + '\n')
                f.close( )			# close : 파일 저장

            driver.back() # 이전 페이지로 돌아가기

            time.sleep(2)

            no += 1

            if no > cnt :
                break

    a += 1
    b = str(a)

    try:
        driver.find_element(By.LINK_TEXT, b).click()
        time.sleep(1)

    except:
        driver.find_element(By.CLASS_NAME, "next_go").click()
        time.sleep(2)

print("요청하신 작업이 모두 완료되었습니다")

## 종합본

In [ ]:
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import random
import time

#Step 3. 수집된 데이터를 저장할 파일 이름 입력 받기
f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본값:C:\\py_temp\\추출 후 파일 저장\\):")
if f_dir == '' :
    f_dir="C:\\py_temp\\추출 후 파일 저장\\"

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s_time = time.time( ) #총 얼마걸리는지 알아냄
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get('https://eungdapso.seoul.go.kr/')
driver.maximize_window()
time.sleep(random.randrange(2,5))

#Step 5. 자동으로 검색어 입력 후 조회하기
driver.find_element(By.XPATH, '//*[@id="content"]/div[2]/a/div/div/div[1]/div[1]').click() #div 중에서 text가 "시장에게 바란다!"라는 걸 찾고 클릭
time.sleep(3)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력 받기
import math
total_cnt = soup_1.find('div','rp_lkment').find('span','rplk_count').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
page_cnt = math.ceil(cnt / 10)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(cnt,page_cnt))
print("\n")

#Step 9. 데이터 수집하기
no2=[] # 게시글 번호 컬럼
title2=[ ] # 게시글 제목 컬럼
opend2=[] # 공개일
story2=[ ] # 상담내역

no = 1 # 게시글 번호 초기값

for x in range(1,page_cnt+1) :
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %a)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    time.sleep(3)
    content_list = soup.find('div','rp_tb_wrap').find('tbody').find_all('tr')

    for i in content_list:
        # 민원 제목 체크하기
        try:
            title=i.find('a','rp_inlink').get_text().strip()
        except :
            continue
        else :
            f = open(ft_name, 'a' , encoding="UTF-8")		#open : 준비 / a 쓰면 추가, w 쓰면 덮어쓰기(w쓰면 100건 크롤링해도 1건만 남음)
            # 1.게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.번호 : %s" %no)
            f.write('\n'+'1.번호:' + str(no))		# write : 쓰기 / 글자, 숫자면 안 맞아서 에러남. 그래서 str 사용하여 글자로 바꿈

            # 2. 민원 제목
            title2.append(title.strip())
            print("2.제목 : %s" %title.strip())
            f.write('\n' + '2.제목:' + title)
            
            # 3. 공개일
            try :
                opend=i.find("td","wid03").find('span').get_text().replace("\n","").strip()
            except :
                opend='공개일이 없습니다'
                opend2.append(opend)
                print("3.공개일 : %s" %opend)
            else :
                opend2.append(opend)
                print("3.공개일 : %s" %opend)
                f.write('\n' + '3.공개일:' + opend)

            time.sleep(1)




            # 상세정보로 들어가야 함         
            driver.find_element(By.XPATH, '//*[@id="content"]/div[2]/div[4]/table/tbody/tr[%s]/td[2]/div/a' %i).click() #url이 없이 자바스크립트로 있기에 XPATH로 가야함
            time.sleep(2)

            # 4. 상담내용
            # 여기서 파싱이 중요(상세정보로 가져왔기에)
            html_1 = driver.page_source
            soup_1 = BeautifulSoup(html_1, 'html.parser')
            time.sleep(2)
            try :
                story=soup_1.find('div','restd_incont').find('span').get_text().strip()
            except :
                story = '상담내용이 없습니다'
                print("4.상담내용 : %s" %story.strip())
                story2.append(story.strip())
            else :
                story2.append(story.strip())
                print("4.상담내용 : %s" %story.strip())
                f.write('\n' + '4.상담내용:' + story + '\n')
                f.close( )			# close : 파일 저장

            driver.back() # 이전 페이지로 돌아가기

            time.sleep(2)

            no += 1

            if no > cnt :
                break

    a += 1
    b = str(a)

    try:
        driver.find_element(By.LINK_TEXT, b).click()
        time.sleep(1)

    except:
        driver.find_element(By.CLASS_NAME, "next_go").click()
        time.sleep(2)

print("요청하신 작업이 모두 완료되었습니다")